# 02 · Do the features actually carry signal?

The decision notebook. For each feature we ask: **does it add predictive value beyond Elo?** If a feature (e.g. squad value, injuries) doesn't lower out-of-sample log-loss over an Elo-only baseline, it isn't worth the collection cost.

Method: mutual information, a permutation-importance model, and an **incremental log-loss** test on a strict **time split** (train pre-2018, test 2018+).

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.width', 120); pd.set_option('display.max_columns', 40)
PROC = ROOT / 'data' / 'processed'

def load_features():
    fp = PROC / 'match_features.parquet'
    if fp.exists():
        return pd.read_parquet(fp)
    csv = PROC / 'match_features_sample.csv'
    if csv.exists():
        print('Using SAMPLE csv (run 02_build_features.py for the full table).')
        return pd.read_csv(csv, parse_dates=['date'])
    raise FileNotFoundError('No processed data. Run scripts/01_download.py then scripts/02_build_features.py')

df = load_features()
print(df.shape, '|', df['date'].min().date(), '->', df['date'].max().date())

from wcpred import datasets, features as F
played = df[df['played']].dropna(subset=['result']).copy()
played['y'] = played['result'].map({'H':0,'D':1,'A':2}).astype(int)
print('played w/ outcome:', len(played))

## A. Mutual information with the outcome
Quick, model-free ranking of which features share information with H/D/A.

In [ ]:
from sklearn.feature_selection import mutual_info_classif
cand = [c for c in F.feature_columns(df) if c in played.columns]
X = played[cand].astype(float).fillna(played[cand].astype(float).median())
mi = pd.Series(mutual_info_classif(X, played['y'], random_state=0), index=cand).sort_values(ascending=False)
print('Mutual information (top 15):'); print(mi.head(15).round(4))
mi.head(15).iloc[::-1].plot(kind='barh', figsize=(6,5)); plt.title('MI with outcome'); plt.tight_layout(); plt.show()

## B. Permutation importance (HistGradientBoosting)
Importance that accounts for interactions and is measured on a held-out period. HistGBM handles NaNs natively, so no imputation distortions.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
tr, te = datasets.time_split(played, '2018-01-01')
Xtr, Xte = tr[cand].astype(float), te[cand].astype(float)
clf = HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=400,
                                     l2_regularization=1.0, random_state=0)
clf.fit(Xtr, tr['y'])
pi = permutation_importance(clf, Xte, te['y'], n_repeats=8, random_state=0, scoring='neg_log_loss')
imp = pd.Series(pi.importances_mean, index=cand).sort_values(ascending=False)
print('Permutation importance (top 15, drop in log-loss):'); print(imp.head(15).round(4))
imp.head(15).iloc[::-1].plot(kind='barh', figsize=(6,5)); plt.title('Permutation importance (neg log-loss)'); plt.tight_layout(); plt.show()

## C. Incremental value over an Elo-only baseline  ←  the real test
Add feature groups one at a time and watch out-of-sample log-loss & RPS. A group that doesn't improve both is a candidate to drop. (Lower is better for both.)

In [ ]:
from sklearn.metrics import log_loss

def rps(probs, y):
    o = np.zeros_like(probs); o[np.arange(len(y)), y] = 1
    cp = np.cumsum(probs, axis=1)[:, :-1]; co = np.cumsum(o, axis=1)[:, :-1]
    return float(((cp - co) ** 2).sum(axis=1).mean() / 1.0)

groups = {
    'elo_only':       ['elo_diff'],
    '+ranking':       ['elo_diff','rank_diff','rank_points_diff'],
    '+form':          ['elo_diff','rank_diff','rank_points_diff','form_ppg_5_diff','form_gd_5_diff','form_ewm_ppg_diff'],
    '+rest/context':  ['elo_diff','rank_diff','rank_points_diff','form_ppg_5_diff','form_gd_5_diff','form_ewm_ppg_diff','rest_days_diff','matches_30d_diff','same_confed'],
}
rows = []
for name, cols in groups.items():
    cols = [c for c in cols if c in tr.columns]
    m = HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=400, random_state=0)
    m.fit(tr[cols].astype(float), tr['y'])
    p = m.predict_proba(te[cols].astype(float))
    rows.append({'feature_set': name, 'n_features': len(cols),
                 'log_loss': round(log_loss(te['y'], p, labels=[0,1,2]), 4),
                 'RPS': round(rps(p, te['y'].to_numpy()), 4)})
print(pd.DataFrame(rows).to_string(index=False))

## D. The advanced features (squad value / injuries)
These exist only for the **modern subset**. If you've run `01_download.py --advanced`, join them and repeat the incremental test on 2015+ matches to see whether they beat Elo+form. If not yet scraped, this cell explains the hook.

In [ ]:
from wcpred import squad
sf = squad.load_squad_features(ROOT / 'data' / 'raw')
if sf is None:
    print('No squad features yet. Run: python scripts/01_download.py --advanced')
    print('Then join sf on home_team/away_team, build *_diff columns, and add them to a 2015+ incremental test.')
else:
    print('Squad features available for', len(sf), 'teams:'); print(sf.head())
    print('\nNext: merge home/away squad_value_eur_m + injured_value_share, form *_diff, and re-run section C on df[df.date>=2015].')

### How to read this
- A feature group earns its place only if it **lowers both log-loss and RPS** out-of-sample vs the row above it.
- Expect Elo to dominate; ranking adds a little; form adds a little more; rest/context is usually marginal. Keep what helps, drop the rest before adding model complexity.
- For squad value/injuries, judge them on the **modern subset** only — that's the only fair test.